# Final Lexicon Playground

Use this notebook to:
- build the **final lexicon** exactly like the app (idioms + merged phrasal verbs),
- inspect entry stats,
- and test whether a phrase is in the lexicon.

## Lexicon Build Report (Quick)

This notebook reflects how the project builds the final phrase lexicon used by MWE search.

### Sources
- `idioms_mcgrawhill.json` (idioms): canonical idiom phrase, patterns, definition.
- `phrasal_verbs_wecan.json` (phrasal verbs): large base list with derivatives, descriptions, examples.
- `phrasal_verbs_semigradsky.json` (phrasal verbs): curated smaller list with explicit separability markers.
- `phrasal_verbs_external_mwe.json` (phrasal verbs): externally mined supplement from MWE datasets.
- `phrasal_verbs_wiktionary_category.json` (phrasal verbs): broad Wiktionary category supplement.

### Build Logic
- Idioms: load canonical phrase + patterns; create normalized text keys for lookup.
- Phrasal verbs: start from WeCan; merge Semigradsky entries by canonical `verb + particle`.
- Then add external MWE entries only if canonical is not already present.
- Then add Wiktionary entries only if canonical is not already present.
- Result: one final phrasal-verb table plus one idiom table, then combined into a unified final lexicon view.

### How Data Is Used
- Stats: counts per source, overlap, separable PV count, and final total entries.
- Browsing: idiom table, phrasal-verb table, and unified lexicon table.
- Query checks: direct canonical matching and idiom normalized-key matching via helper functions (`is_in_lexicon`, `search_lexicon`).

### Notes
- This notebook emphasizes transparent inspection and experimentation.
- Runtime app lookup additionally includes lemma-based fallback; this notebook keeps the core merge/index behavior clear and inspectable.


In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 140)

In [2]:
# Resolve project root so this works when launched from repo root or /experiments
cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'app').exists() else cwd.parent
LEXICON_DIR = ROOT / 'data' / 'lexicons'
EXPORTS_DIR = ROOT / 'data' / 'exports'

IDIOMS_PATH = LEXICON_DIR / 'idioms_mcgrawhill.json'
WECAN_PATH = LEXICON_DIR / 'phrasal_verbs_wecan.json'
SEMI_PATH = LEXICON_DIR / 'phrasal_verbs_semigradsky.json'
EXTERNAL_MWE_PATH = LEXICON_DIR / 'phrasal_verbs_external_mwe.json'
WIKTIONARY_PATH = LEXICON_DIR / 'phrasal_verbs_wiktionary_category.json'

for p in [IDIOMS_PATH, WECAN_PATH, SEMI_PATH, EXTERNAL_MWE_PATH, WIKTIONARY_PATH]:
    print(f'{p.name}:', 'OK' if p.exists() else 'MISSING')

latest_export = None
if EXPORTS_DIR.exists():
    candidates = sorted(EXPORTS_DIR.glob('final_lexicon_export_*.meta.json'))
    latest_export = candidates[-1] if candidates else None

print('latest export meta:', latest_export.name if latest_export else 'none found')


idioms_mcgrawhill.json: OK
phrasal_verbs_wecan.json: OK
phrasal_verbs_semigradsky.json: OK
phrasal_verbs_external_mwe.json: OK
phrasal_verbs_wiktionary_category.json: OK
latest export meta: final_lexicon_export_20260308_140929.meta.json


In [3]:
def load_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def normalize_key(phrase: str) -> str:
    return re.sub(r'[^a-z0-9 ]', '', phrase.lower()).strip()


def parse_semigradsky_verb_field(verb_field: str) -> tuple[str, str, bool]:
    # Mirror app/search/mwe/lexicon.py::_parse_phrasal_verb_entry
    clean = verb_field.replace('+', '').strip()
    separable = '*' in clean
    clean = clean.replace('*', '').strip()
    parts = clean.split()
    if len(parts) >= 2:
        return parts[0], ' '.join(parts[1:]), separable
    return clean, '', False

In [4]:
# Load raw datasets
idioms_raw = load_json(IDIOMS_PATH)['dictionary']
wecan_raw = load_json(WECAN_PATH)
semi_raw = load_json(SEMI_PATH)
external_mwe_raw = load_json(EXTERNAL_MWE_PATH) if EXTERNAL_MWE_PATH.exists() else {}
wiktionary_raw = load_json(WIKTIONARY_PATH) if WIKTIONARY_PATH.exists() else {'phrases': []}

print('Raw idioms (McGrawHill):', len(idioms_raw))
print('Raw phrasal verbs (WeCan):', len(wecan_raw))
print('Raw phrasal verbs (Semigradsky):', len(semi_raw))
print('Raw phrasal verbs (External MWE):', len(external_mwe_raw))
print('Raw phrasal verbs (Wiktionary phrases):', len(wiktionary_raw.get('phrases', [])))


Raw idioms (McGrawHill): 21866
Raw phrasal verbs (WeCan): 3350
Raw phrasal verbs (Semigradsky): 124
Raw phrasal verbs (External MWE): 1089
Raw phrasal verbs (Wiktionary phrases): 5068


In [5]:
# Build idiom lookup like the app (normalized phrase + patterns)
idiom_lookup = {}
idiom_rows = []

for entry in idioms_raw:
    phrase = entry.get('phrase', '')
    if not phrase:
        continue

    patterns = entry.get('patterns', [phrase])
    definition = re.sub(r'[_]', '', entry.get('definition', '')).strip()

    row = {
        'type': 'idiom',
        'canonical': phrase,
        'definition': definition,
        'patterns_count': len(patterns),
        'normalized': normalize_key(phrase),
    }
    idiom_rows.append(row)

    idiom_entry = {
        'canonical_form': phrase,
        'patterns': patterns,
        'definition': definition,
    }

    idiom_lookup[normalize_key(phrase)] = idiom_entry
    for p in patterns:
        idiom_lookup[normalize_key(p)] = idiom_entry

idiom_df = pd.DataFrame(idiom_rows).drop_duplicates(subset=['canonical']).sort_values('canonical').reset_index(drop=True)

print('Unique idiom canonicals:', len(idiom_df))
print('Idiom lookup keys (phrase + patterns):', len(idiom_lookup))

Unique idiom canonicals: 21253
Idiom lookup keys (phrase + patterns): 39919


In [6]:
# Build merged phrasal verbs like the app (lexicon.py)
pv_dict = {}

# Source 1: wecan
for phrase, data in wecan_raw.items():
    parts = phrase.split()
    verb = parts[0] if parts else phrase
    particle = ' '.join(parts[1:]) if len(parts) >= 2 else ''

    pv_dict[phrase] = {
        'canonical': phrase,
        'verb': verb,
        'particle': particle,
        'separable': False,
        'definition': '; '.join(data.get('descriptions', [])),
        'examples': data.get('examples', []),
        'derivatives': data.get('derivatives', []),
        'from_wecan': True,
        'from_semigradsky': False,
        'from_external_mwe': False,
        'from_wiktionary': False,
    }

# Source 2: semigradsky (merge/override separability)
semi_overlap = 0
for entry in semi_raw:
    verb_field = entry.get('verb', '')
    verb, particle, separable = parse_semigradsky_verb_field(verb_field)
    canonical = f'{verb} {particle}'.strip()

    if canonical in pv_dict:
        semi_overlap += 1
        pv_dict[canonical]['separable'] = separable
        pv_dict[canonical]['from_semigradsky'] = True
    else:
        pv_dict[canonical] = {
            'canonical': canonical,
            'verb': verb,
            'particle': particle,
            'separable': separable,
            'definition': entry.get('definition', ''),
            'examples': entry.get('examples', []),
            'derivatives': [],
            'from_wecan': False,
            'from_semigradsky': True,
            'from_external_mwe': False,
            'from_wiktionary': False,
        }

# Source 3: external MWE supplement (add only if new)
external_added = 0
external_overlap = 0
for phrase, data in external_mwe_raw.items():
    parts = phrase.split()
    if len(parts) < 2:
        continue
    verb = parts[0]
    particle = ' '.join(parts[1:])
    canonical = f'{verb} {particle}'.strip()

    if canonical in pv_dict:
        external_overlap += 1
        continue

    pv_dict[canonical] = {
        'canonical': canonical,
        'verb': verb,
        'particle': particle,
        'separable': False,
        'definition': '; '.join(data.get('descriptions', [])),
        'examples': data.get('examples', []),
        'derivatives': data.get('derivatives', []),
        'from_wecan': False,
        'from_semigradsky': False,
        'from_external_mwe': True,
        'from_wiktionary': False,
    }
    external_added += 1

# Source 4: wiktionary supplement (add only if new)
wiktionary_added = 0
wiktionary_overlap = 0
for phrase in wiktionary_raw.get('phrases', []):
    parts = phrase.split()
    if len(parts) < 2:
        continue
    verb = parts[0]
    particle = ' '.join(parts[1:])
    canonical = f'{verb} {particle}'.strip()

    if canonical in pv_dict:
        wiktionary_overlap += 1
        continue

    pv_dict[canonical] = {
        'canonical': canonical,
        'verb': verb,
        'particle': particle,
        'separable': False,
        'definition': '',
        'examples': [],
        'derivatives': [],
        'from_wecan': False,
        'from_semigradsky': False,
        'from_external_mwe': False,
        'from_wiktionary': True,
    }
    wiktionary_added += 1

pv_df = pd.DataFrame(pv_dict.values()).sort_values('canonical').reset_index(drop=True)
pv_df['derivatives_count'] = pv_df['derivatives'].map(lambda x: len(x) if isinstance(x, list) else 0)

print('Final phrasal verb canonicals:', len(pv_df))
print('Semigradsky overlap (merged into existing):', semi_overlap)
print('External MWE added:', external_added, '| overlap:', external_overlap)
print('Wiktionary added:', wiktionary_added, '| overlap:', wiktionary_overlap)
print('Final separable phrasal verbs:', int(pv_df['separable'].sum()))


Final phrasal verb canonicals: 6691
Semigradsky overlap (merged into existing): 120
External MWE added: 729 | overlap: 360
Wiktionary added: 2608 | overlap: 2460
Final separable phrasal verbs: 11


In [7]:
# Final lexicon stats summary
final_lexicon_df = pd.concat([
    idiom_df[['type', 'canonical', 'definition']],
    pv_df.assign(type='phrasal_verb')[['type', 'canonical', 'definition']],
], ignore_index=True)

summary = pd.DataFrame([
    {'metric': 'idiom_canonicals', 'value': len(idiom_df)},
    {'metric': 'idiom_lookup_keys', 'value': len(idiom_lookup)},
    {'metric': 'phrasal_verb_canonicals', 'value': len(pv_df)},
    {'metric': 'phrasal_verbs_from_wecan', 'value': int(pv_df['from_wecan'].sum())},
    {'metric': 'phrasal_verbs_from_semigradsky', 'value': int(pv_df['from_semigradsky'].sum())},
    {'metric': 'phrasal_verbs_from_external_mwe', 'value': int(pv_df['from_external_mwe'].sum())},
    {'metric': 'phrasal_verbs_from_wiktionary', 'value': int(pv_df['from_wiktionary'].sum())},
    {'metric': 'phrasal_verbs_separable', 'value': int(pv_df['separable'].sum())},
    {'metric': 'final_total_canonicals', 'value': len(final_lexicon_df)},
])

summary


,metric,value
0,idiom_canonicals,21253
1,idiom_lookup_keys,39919
2,phrasal_verb_canonicals,6691
3,phrasal_verbs_from_wecan,3350
4,phrasal_verbs_from_semigradsky,68
5,phrasal_verbs_from_external_mwe,729
6,phrasal_verbs_from_wiktionary,2608
7,phrasal_verbs_separable,11
8,final_total_canonicals,27944


## Browse entries
Use these cells to inspect idioms, phrasal verbs, and the merged final lexicon.

In [8]:
idiom_df.head(20)

,type,canonical,definition,patterns_count,normalized
0,idiom,(Are you) doing okay?,"1. How are you? Mary: Doing okay? Bill: You bet! How are you? Bill: Hey, man! Are you doing okay? Tom: Sure thing! And you? 2. How are ...",2,are you doing okay
1,idiom,(Are you) feeling okay?,"Do you feel well? Tom: Are you feeling okay? Bill: Oh, fair to middling. Susan: Are you feeling okay? Mary: I’m still a little dizzy, ...",2,are you feeling okay
2,idiom,(Are you) going my way?,"If you are traveling in the direction of my destination, could I please go with you or can I have a ride in your car? Mary: Are you goi...",2,are you going my way
3,idiom,(Are you) leaving so soon?,a polite inquiry made to a guest who has announced a departure. (Appropriate only for the first few guests to leave. It would seem sarca...,2,are you leaving so soon
4,idiom,(Are you) ready for this?,"a way of presenting a piece of news or information that is expected to excite or surprise the person spoken to. Tom: Boy, do I have som...",2,are you ready for this
5,idiom,(Are you) ready to order?,Would you please tell me what you want as your meal? (A standard phrase used in eating establishments to find out what a customer wants ...,2,are you ready to order
6,idiom,(Are you) sorry you asked?,"Now that you have heard (the unpleasant answer), do you regret having asked the question? (Compare this with You’ll be sorry you asked.)...",2,are you sorry you asked
7,idiom,(Are) things getting you down?,"Are everyday issues bothering you? Jane: Gee, Mary, you look sad. Are things getting you down? Tom: What’s the matter, Bob? Things gett...",2,are things getting you down
8,idiom,(Aw) shucks!,"Rur. Gosh!; a mild oath. Shucks, ma’am. It wasn’t anything at all. Aw shucks, I ain’t never been this close to a woman before.",2,aw shucks
9,idiom,(Could I) buy you a drink?,1. Lit. Could I purchase a drink for you? (An offer by one person—usually in a bar—to buy a drink for another. Then the two will drink t...,2,could i buy you a drink


In [9]:
pv_df[['canonical', 'verb', 'particle', 'separable', 'from_wecan', 'from_semigradsky', 'from_external_mwe', 'from_wiktionary', 'definition']].head(20)


,canonical,verb,particle,separable,from_wecan,from_semigradsky,from_external_mwe,from_wiktionary,definition
0,'fess up,'fess,up,False,False,False,False,True,
1,ab off,ab,off,False,False,False,False,True,
2,abate of,abate,of,False,False,False,False,True,
3,abide by,abide,by,False,True,False,False,False,"to follow a rule, decision, or instruction"
4,absorb oneself in,absorb,oneself in,False,False,False,False,True,
5,abstract away from,abstract,away from,False,False,False,False,True,
6,abut on,abut,on,False,False,False,False,True,
7,accord to,accord,to,False,False,False,True,False,External MWE candidate from dimsum
8,accord with,accord,with,False,True,False,False,False,to agree with or be the same as something else
9,account for,account,for,False,True,False,False,False,"if someone is accounted for, you know where they are and so do not worry that they are not where they should be; to be the reason why so..."


In [10]:
final_lexicon_df.sort_values(['type', 'canonical']).head(30)

,type,canonical,definition
0,idiom,(Are you) doing okay?,"1. How are you? Mary: Doing okay? Bill: You bet! How are you? Bill: Hey, man! Are you doing okay? Tom: Sure thing! And you? 2. How are ..."
1,idiom,(Are you) feeling okay?,"Do you feel well? Tom: Are you feeling okay? Bill: Oh, fair to middling. Susan: Are you feeling okay? Mary: I’m still a little dizzy, ..."
2,idiom,(Are you) going my way?,"If you are traveling in the direction of my destination, could I please go with you or can I have a ride in your car? Mary: Are you goi..."
3,idiom,(Are you) leaving so soon?,a polite inquiry made to a guest who has announced a departure. (Appropriate only for the first few guests to leave. It would seem sarca...
4,idiom,(Are you) ready for this?,"a way of presenting a piece of news or information that is expected to excite or surprise the person spoken to. Tom: Boy, do I have som..."
5,idiom,(Are you) ready to order?,Would you please tell me what you want as your meal? (A standard phrase used in eating establishments to find out what a customer wants ...
6,idiom,(Are you) sorry you asked?,"Now that you have heard (the unpleasant answer), do you regret having asked the question? (Compare this with You’ll be sorry you asked.)..."
7,idiom,(Are) things getting you down?,"Are everyday issues bothering you? Jane: Gee, Mary, you look sad. Are things getting you down? Tom: What’s the matter, Bob? Things gett..."
8,idiom,(Aw) shucks!,"Rur. Gosh!; a mild oath. Shucks, ma’am. It wasn’t anything at all. Aw shucks, I ain’t never been this close to a woman before."
9,idiom,(Could I) buy you a drink?,1. Lit. Could I purchase a drink for you? (An offer by one person—usually in a bar—to buy a drink for another. Then the two will drink t...


## Phrase check + search helpers
`is_in_lexicon` gives a direct yes/no with details.
`search_lexicon` helps exploratory contains/prefix search.

In [11]:
idiom_canonical_set = set(idiom_df['canonical'].str.lower())
pv_canonical_set = set(pv_df['canonical'].str.lower())
all_canonical_df = pd.concat([
    idiom_df[['type', 'canonical', 'definition']],
    pv_df.assign(type='phrasal_verb')[['type', 'canonical', 'definition']],
], ignore_index=True)
all_canonical_df['canonical_lower'] = all_canonical_df['canonical'].str.lower()


def is_in_lexicon(query: str) -> dict:
    q_norm = normalize_key(query)
    q_low = query.strip().lower()

    idiom_lookup_match = idiom_lookup.get(q_norm)
    idiom_canonical_match = q_low in idiom_canonical_set
    pv_canonical_match = q_low in pv_canonical_set

    return {
        'query': query,
        'normalized_query': q_norm,
        'idiom_lookup_match': idiom_lookup_match is not None,
        'idiom_canonical_match': idiom_canonical_match,
        'phrasal_verb_canonical_match': pv_canonical_match,
        'idiom_canonical_form': idiom_lookup_match['canonical_form'] if idiom_lookup_match else None,
        'idiom_definition': idiom_lookup_match['definition'] if idiom_lookup_match else None,
    }


def search_lexicon(query: str, mode: str = 'contains', only_type: str | None = None, limit: int = 25) -> pd.DataFrame:
    q = query.strip().lower()
    df = all_canonical_df.copy()

    if only_type in {'idiom', 'phrasal_verb'}:
        df = df[df['type'] == only_type]

    if mode == 'exact':
        out = df[df['canonical_lower'] == q]
    elif mode == 'prefix':
        out = df[df['canonical_lower'].str.startswith(q)]
    else:
        out = df[df['canonical_lower'].str.contains(re.escape(q), regex=True)]

    return out[['type', 'canonical', 'definition']].sort_values(['type', 'canonical']).head(limit).reset_index(drop=True)

In [12]:
# Examples
is_in_lexicon('turn the corner')

{'query': 'turn the corner',
 'normalized_query': 'turn the corner',
 'idiom_lookup_match': True,
 'idiom_canonical_match': True,
 'phrasal_verb_canonical_match': False,
 'idiom_canonical_form': 'turn the corner',
 'idiom_definition': 'Fig. to pass a critical point in a process. The patient turned the corner last night. She should begin to show improvement now.  The project has turned the corner. The rest should be easy.'}

In [18]:
search_lexicon('go well', mode='contains', limit=20)

,type,canonical,definition
0,idiom,go well with someone or something,[for something] to proceed nicely for someone or something. I hope things are going well with you. Things are going very well with the...


In [14]:
search_lexicon('give up', mode='exact')

,type,canonical,definition
0,idiom,give up,to quit; to quit trying. I give up! I won’t press this further. Are you going to give up or keep fighting?
1,phrasal_verb,give up,"if you give something up as lost, you believe that you will not find it and you stop looking for it; if you give yourself up, you allow ..."


## Advanced Variant Search (placeholders like somebody)
Use this when phrases may be stored as templates/variants (e.g., `someone`, `sb`, `oneself`) rather than exact canonical text.

In [15]:
# Build richer search tables from source-specific variant fields
idiom_variant_rows = []
for e in idioms_raw:
    canonical = e.get('phrase', '')
    definition = re.sub(r'[_]', '', e.get('definition', '')).strip()
    for p in e.get('patterns', []) or []:
        idiom_variant_rows.append({
            'type': 'idiom',
            'canonical': canonical,
            'variant': p,
            'variant_source': 'pattern',
            'definition': definition,
        })

pv_variant_rows = []
for _, row in pv_df.iterrows():
    canonical = row['canonical']
    definition = row.get('definition', '')

    pv_variant_rows.append({
        'type': 'phrasal_verb',
        'canonical': canonical,
        'variant': canonical,
        'variant_source': 'canonical',
        'definition': definition,
    })

    for d in row.get('derivatives', []) or []:
        pv_variant_rows.append({
            'type': 'phrasal_verb',
            'canonical': canonical,
            'variant': d,
            'variant_source': 'derivative',
            'definition': definition,
        })

variant_df = pd.DataFrame(idiom_variant_rows + pv_variant_rows).dropna(subset=['variant']).copy()


def _normalize_placeholder_text(s: str) -> str:
    s = s.lower().strip()
    # unify common learner placeholders
    s = re.sub(r'(somebody|someone|some one|sb)', '__person__', s)
    s = re.sub(r'(something|sth|some thing)', '__thing__', s)
    s = re.sub(r'(oneself|yourself|myself|himself|herself|itself|ourselves|yourselves|themselves)', '__self__', s)
    s = re.sub(r'[^a-z0-9_ ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


variant_df['variant_norm'] = variant_df['variant'].map(_normalize_placeholder_text)


def search_variants(query: str, mode: str = 'contains', only_type: str | None = None, limit: int = 30) -> pd.DataFrame:
    q = _normalize_placeholder_text(query)
    df = variant_df
    if only_type in {'idiom', 'phrasal_verb'}:
        df = df[df['type'] == only_type]

    if mode == 'exact':
        out = df[df['variant_norm'] == q]
    elif mode == 'prefix':
        out = df[df['variant_norm'].str.startswith(q)]
    else:
        out = df[df['variant_norm'].str.contains(re.escape(q), regex=True)]

    return (
        out[['type', 'canonical', 'variant', 'variant_source', 'definition']]
        .drop_duplicates()
        .sort_values(['type', 'canonical', 'variant_source', 'variant'])
        .head(limit)
        .reset_index(drop=True)
    )

print('Variant rows:', len(variant_df))


Variant rows: 45499
